# 24 — Prompt Versioning, Experimentation, and Release Engineering

## Scenario
We have a stable prompt in production (`v1.0`). A developer thinks they've written a better one (`v1.1`).

**The Problem:** If we just replace `v1.0` with `v1.1`, and `v1.1` hallucinates, all our users are impacted instantly.

**The Solution:** We use a **Canary Release**. We route 90% of traffic to `v1.0`, and 10% to `v1.1`. If `v1.1` throws errors, we automatically trigger a **Rollback**.

In [ ]:
import os
import random
from typing import Dict
from pydantic import BaseModel, Field
from google import genai
from google.genai import types

client = genai.Client()
MODEL_ID = 'gemini-2.5-flash'


## Step 1: The Artifacts

We define our stable `v1.0` and our risky new `v1.1`.

In [ ]:
class TranslationResult(BaseModel):
    french: str = Field(description="The french translation")

# Production (Stable)
prompt_v1_0 = "Translate the following text to French. Be literal."

# Candidate (Risky - The developer told it to be creative, which might break the schema or expectations)
prompt_v1_1 = "Translate the following text to French, but make it sound like a pirate."


## Step 2: The Canary Router and Evaluation Gate

We build a router that sends 10% of traffic to the candidate. We also build an evaluation gate that tracks the error rate of the canary.

In [ ]:
class ReleaseManager:
    def __init__(self):
        self.canary_traffic_percent = 10
        self.canary_errors = 0
        self.canary_total = 0
        self.max_error_rate = 0.2 # If more than 20% of canary requests fail, rollback
        self.rollback_triggered = False
        
    def route_request(self, text: str) -> str:
        if self.rollback_triggered:
            return self.execute(prompt_v1_0, text, "v1.0 (Post-Rollback)")
            
        # Roll the dice for Canary
        if random.randint(1, 100) <= self.canary_traffic_percent:
            self.canary_total += 1
            try:
                result = self.execute(prompt_v1_1, text, "v1.1 (Canary)")
                return result
            except Exception as e:
                self.canary_errors += 1
                print(f"[ERROR] Canary v1.1 failed: {e}")
                self.check_rollback()
                # Fallback to stable for the user
                return self.execute(prompt_v1_0, text, "v1.0 (Fallback)")
        else:
            return self.execute(prompt_v1_0, text, "v1.0 (Production)")
            
    def execute(self, prompt: str, text: str, version_tag: str) -> str:
        # We simulate that the 'pirate' prompt confuses the model so much it breaks the JSON schema
        if prompt == prompt_v1_1 and random.choice([True, False]):
            raise ValueError("Schema Violation: Model output invalid JSON due to pirate speak.")
            
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=f"{prompt}\nText: {text}",
            config=types.GenerateContentConfig(
                temperature=0.0,
                response_mime_type="application/json",
                response_schema=TranslationResult,
            )
        )
        return f"[{version_tag}] {TranslationResult.model_validate_json(response.text).french}"

    def check_rollback(self):
        if self.canary_total >= 3: # Wait for at least 3 canary requests before judging
            error_rate = self.canary_errors / self.canary_total
            if error_rate > self.max_error_rate:
                print(f"\n🚨 [ALERT] Canary Error Rate ({error_rate*100:.1f}%) exceeded threshold ({self.max_error_rate*100:.1f}%). Triggering Automated Rollback to v1.0! 🚨\n")
                self.rollback_triggered = True


## Step 3: Running Traffic

We send 30 requests through the router. Watch as the Canary receives some traffic, fails, and triggers the rollback.

In [ ]:
random.seed(42) # For deterministic simulation
manager = ReleaseManager()

for i in range(1, 31):
    print(f"Request {i:02d}: ", end="")
    result = manager.route_request("Hello world")
    print(result)


## Conclusion

If we had deployed `v1.1` to 100% of users immediately, the application would have experienced a massive outage.

Because we used a **Canary Release** and an automated **Evaluation Gate**, only a tiny fraction of users experienced the error, the system automatically fell back to the stable version for those users, and the entire deployment was rolled back before significant damage was done.